In [21]:
!pip -q install gradio requests

In [22]:
import re
import requests
import gradio as gr
from getpass import getpass

# Enter your Hugging Face access token securely
HF_TOKEN = getpass("Enter your Hugging Face token: ")

# AI model configuration
HF_MODEL = "openai/gpt-oss-120b:fastest"

# Hugging Face chat completion API
API_URL = "https://router.huggingface.co/v1/chat/completions"

print("Libraries imported successfully!")
print("AI model configuration completed.")

Enter your Hugging Face token: ··········
Libraries imported successfully!
AI model configuration completed.


In [23]:
def build_prompt(topic, tone, content_type, length):
    if content_type == "Blog Post":
        return f"""
        Write a {length.lower()} blog post about {topic}.
        Use a {tone.lower()} tone.
        Include a suitable title, introduction, main points,
        practical examples, and a conclusion.
        """

    elif content_type == "Social Media Caption":
        return f"""
        Write 3 engaging social media captions about {topic}.
        Use a {tone.lower()} tone.
        Include relevant hashtags and emojis where appropriate.
        """

    else:
        return f"""
        Summarize {topic} in a {tone.lower()} tone.
        Explain the main ideas clearly and concisely.
        """

In [24]:

def fallback_content(topic, tone, content_type, length):
    if content_type == "Blog Post":
        return f"""# {topic.title()}

## Introduction
{topic} is an important subject in today's world.
Understanding it helps us explore new ideas and opportunities.

## Main Points
A useful way to understand {topic} is to learn its fundamentals,
identify its benefits, and explore practical applications.
Consistent learning and real-world examples make the topic easier
to understand.

## Practical Applications
- Learn the basic concepts of {topic}.
- Explore examples and practical uses.
- Apply the knowledge to real-world situations.

## Conclusion
{topic} offers opportunities to learn and improve.
With the right approach and consistent effort, we can better
understand its importance.
"""

    elif content_type == "Social Media Caption":
        return f"""1. Discover something new about {topic}! 🚀
Keep learning and growing every day. #Learning #Growth

2. Small steps can make a big difference.
Let's explore {topic} together! ✨ #Inspiration

3. Curious about {topic}? Start today and see where
your curiosity takes you! 💡 #Explore #Motivation
"""

    else:
        return f"""Summary of {topic}

{topic} can be understood by exploring its key concepts,
practical applications, benefits, and challenges.
Learning the fundamentals and using reliable information
can help develop a clearer understanding of the subject.
"""


def generate_content(topic, tone, content_type, length):
    topic = (topic or "").strip()

    if not topic:
        return "", 0, 0, "Please enter a topic."

    prompt = build_prompt(topic, tone, content_type, length)
    generated_text = ""
    status = ""

    if HF_TOKEN.strip():
        try:
            response = requests.post(
                API_URL,
                headers={
                    "Authorization": f"Bearer {HF_TOKEN.strip()}",
                    "Content-Type": "application/json"
                },
                json={
                    "model": HF_MODEL,
                    "messages": [
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],
                    "max_tokens": 500,
                    "temperature": 0.7
                },
                timeout=90
            )

            if response.ok:
                data = response.json()
                generated_text = (
                    data["choices"][0]["message"]["content"].strip()
                )

                if generated_text:
                    status = "Generated using Hugging Face AI."
                else:
                    status = "The AI returned empty content. Using fallback."

            else:
                print("HTTP Status:", response.status_code)
                print("API Error Details:", response.text)

                status = (
                    f"AI API error ({response.status_code}). "
                    "Check the cell output for details. Using fallback."
                )

        except (
            requests.RequestException,
            KeyError,
            ValueError,
            IndexError
        ) as error:
            print("Error:", str(error))
            status = "AI request failed. Using fallback content."

    else:
        status = "No API token provided. Using fallback content."

    if not generated_text:
        generated_text = fallback_content(
            topic, tone, content_type, length
        )

    word_count = len(re.findall(r"\b[\w'-]+\b", generated_text))
    character_count = len(generated_text)

    return generated_text, word_count, character_count


In [25]:
def refine_content(
    topic, tone, content_type, length, current_text, instruction
):
    if not current_text:
        return generate_content(topic, tone, content_type, length)

    if not instruction.strip():
        words = len(re.findall(r"\b[\w'-]+\b", current_text))
        return (
            current_text,
            words,
            len(current_text),
            "Enter an instruction to refine the content."
        )

    if not HF_TOKEN.strip():
        revised = (
            current_text
            + "\n\nRefinement request: "
            + instruction.strip()
            + "\n\nNote: Add an API token to enable AI-based refinement."
        )

        words = len(re.findall(r"\b[\w'-]+\b", revised))
        return (
            revised,
            words,
            len(revised),
            "Refinement instruction added; no AI token configured."
        )

    prompt = f"""
    Improve the following content based on the instruction.

    Topic: {topic}
    Tone: {tone}
    Content type: {content_type}
    Instruction: {instruction}

    Existing content:
    {current_text}

    Return only the improved content.
    """

    try:
        response = requests.post(
            API_URL,
            headers={
                "Authorization": f"Bearer {HF_TOKEN.strip()}",
                "Content-Type": "application/json"
            },
            json={
                "model": HF_MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 600,
                "temperature": 0.7
            },
            timeout=90
        )

        response.raise_for_status()
        revised = response.json()["choices"][0]["message"]["content"].strip()

        return (
            revised,
            len(re.findall(r"\b[\w'-]+\b", revised)),
            len(revised),
            "Content refined using Hugging Face AI."
        )

    except (requests.RequestException, KeyError, ValueError, IndexError):
        return (
            current_text,
            len(re.findall(r"\b[\w'-]+\b", current_text)),
            len(current_text),
            "Refinement failed. Please try again."
        )

In [26]:

with gr.Blocks(title="AI Content Generator") as demo:

    gr.Markdown(
        """
        # AI Content Generator

        Generate blog posts, social media captions, and text summaries
        using artificial intelligence.
        """
    )

    with gr.Row():
        topic = gr.Textbox(
            label="Enter Topic",
            placeholder="Example: Artificial Intelligence in Education",
            lines=2
        )

        content_type = gr.Dropdown(
            choices=[
                "Blog Post",
                "Social Media Caption",
                "Text Summary"
            ],
            value="Blog Post",
            label="Content Type"
        )

    with gr.Row():
        tone = gr.Dropdown(
            choices=[
                "Professional",
                "Friendly",
                "Creative",
                "Formal",
                "Casual",
                "Persuasive"
            ],
            value="Professional",
            label="Writing Tone"
        )

        length = gr.Dropdown(
            choices=[
                "Short",
                "Medium",
                "Detailed"
            ],
            value="Medium",
            label="Content Length"
        )

    generate_button = gr.Button(
        "Generate Content",
        variant="primary"
    )

    output = gr.Textbox(
        label="Generated Content",
        lines=18
    )

    with gr.Row():
        word_count = gr.Number(
            label="Word Count",
            precision=0
        )

        character_count = gr.Number(
            label="Character Count",
            precision=0
        )

    generate_button.click(
        fn=generate_content,
        inputs=[
            topic,
            tone,
            content_type,
            length
        ],
        outputs=[
            output,
            word_count,
            character_count
        ]
    )

demo.launch(
    share=True,
    theme=gr.themes.Soft()
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f6fd45dd823d68cb5e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
